# 02 - Keyword Heuristic Spoiler Detection (Signal 1)

This is our **baseline** signal. Before applying any ML, we test how far simple
pattern matching gets us. This establishes a floor that ML models must beat to
justify their complexity.

The detector looks for two types of patterns:
1. **Meta-warnings**: comments warning the trailer is spoilery ("shows too much", "spoiler alert")
2. **Plot-reveal indicators**: comments hinting at plot details ("he dies", "the twist is")

In [1]:
import sys
sys.path.insert(0, '..')  # so we can import from src/

from src.model.keyword_detector import score_comment, score_comments

## 1. Test with Individual Comments

Let's feed in realistic YouTube-style comments and see how the detector scores them.
We'll include obvious spoiler warnings, subtle ones, and non-spoiler comments.

In [2]:
# Test cases: (comment, expected_flag)
test_comments = [
    # Should be flagged (meta-warnings about the trailer)
    ("SPOILER ALERT! Don't watch this trailer if you haven't seen the movie!", True),
    ("This trailer shows too much. They basically gave away the whole plot.", True),
    ("Why did they show the final battle in the trailer?? Ruined it for me", True),
    ("They should not have shown that scene at 1:42. Major spoiler.", True),
    ("Don't watch this trailer, it reveals everything", True),
    
    # Should be flagged (plot reveals in the comment itself)
    ("I can't believe he dies at the end, and they just showed it in the trailer", True),
    ("So the villain is actually his father?? The trailer just revealed the plot twist", True),
    
    # Should NOT be flagged (general comments, no spoiler signal)
    ("This looks amazing! Can't wait to see it!", False),
    ("The CGI in this trailer is incredible", False),
    ("Who's here after watching the movie? It was even better than the trailer!", False),
    ("First!", False),
    ("The soundtrack is fire", False),
    ("I've watched this trailer 100 times already", False),
    ("This is going to be the best movie of 2025", False),
    ("Anyone know the song at 2:15?", False),
]

print(f"{'COMMENT':<70} {'SCORE':>6} {'FLAG':>5} {'EXPECTED':>9} {'MATCH':>6}")
print("=" * 100)

correct = 0
for comment, expected in test_comments:
    result = score_comment(comment)
    match = result.is_flagged == expected
    correct += int(match)
    
    # Truncate long comments for display
    display = comment[:67] + "..." if len(comment) > 70 else comment
    status = "OK" if match else "MISS"
    print(f"{display:<70} {result.score:>6.2f} {str(result.is_flagged):>5} {str(expected):>9} {status:>6}")

print(f"\nAccuracy on test set: {correct}/{len(test_comments)} ({correct/len(test_comments)*100:.0f}%)")

COMMENT                                                                 SCORE  FLAG  EXPECTED  MATCH
SPOILER ALERT! Don't watch this trailer if you haven't seen the movie!   1.00  True      True     OK
This trailer shows too much. They basically gave away the whole plot.    1.00  True      True     OK
Why did they show the final battle in the trailer?? Ruined it for me     0.85  True      True     OK
They should not have shown that scene at 1:42. Major spoiler.            0.95  True      True     OK
Don't watch this trailer, it reveals everything                          1.00  True      True     OK
I can't believe he dies at the end, and they just showed it in the ...   0.60  True      True     OK
So the villain is actually his father?? The trailer just revealed t...   1.00  True      True     OK
This looks amazing! Can't wait to see it!                                0.00 False     False     OK
The CGI in this trailer is incredible                                    0.00 False     Fal

## 2. Inspect Pattern Matches

For flagged comments, let's see exactly which patterns triggered and why.

In [3]:
# Show detailed results for each flagged comment
for comment, _ in test_comments:
    result = score_comment(comment)
    if result.is_flagged:
        print(f"Comment: \"{comment[:80]}...\"" if len(comment) > 80 else f"Comment: \"{comment}\"")
        print(f"  Score:    {result.score}")
        print(f"  Patterns: {', '.join(result.matched_patterns)}")
        print()

Comment: "SPOILER ALERT! Don't watch this trailer if you haven't seen the movie!"
  Score:    1.0
  Patterns: spoiler mention, don't watch trailer

Comment: "This trailer shows too much. They basically gave away the whole plot."
  Score:    1.0
  Patterns: shows too much, gave away, whole movie/plot

Comment: "Why did they show the final battle in the trailer?? Ruined it for me"
  Score:    0.85
  Patterns: ruined, why did they show

Comment: "They should not have shown that scene at 1:42. Major spoiler."
  Score:    0.95
  Patterns: spoiler mention, shouldn't have shown, timestamp reference

Comment: "Don't watch this trailer, it reveals everything"
  Score:    1.0
  Patterns: don't watch trailer, reveals too much

Comment: "I can't believe he dies at the end, and they just showed it in the trailer"
  Score:    0.6
  Patterns: death reference

Comment: "So the villain is actually his father?? The trailer just revealed the plot twist"
  Score:    1.0
  Patterns: reveals too much, plot 

## 3. Batch Analysis with Simulated Comment Section

Simulate a realistic trailer comment section (mix of spoiler and non-spoiler comments)
and run the batch scorer to see aggregate statistics.

In [4]:
# Simulated comment section from a hypothetical spoilery trailer
simulated_comments = [
    "This movie looks incredible! Day one for me.",
    "The visuals are stunning",
    "SPOILER ALERT they literally showed the ending in this trailer",
    "Who else got chills watching this?",
    "They gave away the whole movie in 2 minutes smh",
    "Best trailer of 2025",
    "Don't watch this trailer if you want to enjoy the movie. Shows way too much.",
    "First comment!",
    "The acting looks phenomenal",
    "Why did they show the hero dying at 1:45? Totally ruined the movie for me",
    "Can't wait for this!",
    "This trailer reveals the entire plot twist. What were they thinking?",
    "I've been waiting for this sequel for 10 years",
    "The soundtrack is amazing, anyone know the artist?",
    "So turns out that the mentor is actually the villain?? Thanks trailer.",
    "Looks mid tbh",
    "They should not have put that scene in the trailer",
    "This is going to break box office records",
    "Another day, another trailer that spoils the entire movie",
    "The cinematography looks gorgeous",
]

results = score_comments(simulated_comments)

print("BATCH ANALYSIS SUMMARY")
print("=" * 50)
print(f"Total comments analyzed: {results['total_comments']}")
print(f"Flagged as spoiler:     {results['flagged_count']} ({results['flagged_percentage']}%)")
print(f"Average score:          {results['avg_score']}")
print(f"Max score:              {results['max_score']}")
print()
print("FLAGGED COMMENTS (sorted by confidence):")
print("-" * 50)
for r in results['flagged_comments']:
    print(f"  [{r.score:.2f}] {r.text[:75]}")
    print(f"         Patterns: {', '.join(r.matched_patterns)}")

BATCH ANALYSIS SUMMARY
Total comments analyzed: 20
Flagged as spoiler:     8 (40.0%)
Average score:          0.297
Max score:              0.95

FLAGGED COMMENTS (sorted by confidence):
--------------------------------------------------
  [0.95] Why did they show the hero dying at 1:45? Totally ruined the movie for me
         Patterns: ruined, why did they show, timestamp reference
  [0.90] Don't watch this trailer if you want to enjoy the movie. Shows way too much
         Patterns: don't watch trailer
  [0.85] They gave away the whole movie in 2 minutes smh
         Patterns: gave away, whole movie/plot
  [0.75] This trailer reveals the entire plot twist. What were they thinking?
         Patterns: entire movie/plot, plot twist mention
  [0.70] SPOILER ALERT they literally showed the ending in this trailer
         Patterns: spoiler mention
  [0.70] They should not have put that scene in the trailer
         Patterns: shouldn't have shown
  [0.60] Another day, another trailer that s

## 4. Strengths and Limitations

Document what the keyword detector handles well and where it falls short.
This informs what the ML signals (zero-shot, custom model) need to improve on.

In [5]:
# Edge cases the keyword detector might struggle with
edge_cases = [
    # Sarcasm / ambiguity
    "Oh great, another trailer that 'doesn't spoil anything' (wink wink)",
    # Indirect spoiler reference
    "If you've read the book, you know exactly what scene they just showed",
    # Non-English spoiler slang
    "bruh they literally showed THAT scene from the comics",
    # False positive: uses spoiler-adjacent words but isn't about spoilers
    "This movie is going to be a killer at the box office",
    # Misspellings
    "this traler spoild the hole movie",
    # Vague but meaningful to fans
    "They actually showed the snap... why",
]

print("EDGE CASES -- where keyword detection struggles:")
print("=" * 60)
for comment in edge_cases:
    result = score_comment(comment)
    status = "FLAGGED" if result.is_flagged else "MISSED"
    print(f"  [{status:>7}] (score: {result.score:.2f}) {comment}")
    if result.matched_patterns:
        print(f"           Patterns: {', '.join(result.matched_patterns)}")

print()
print("TAKEAWAY: The keyword detector catches explicit warnings well but misses")
print("sarcasm, indirect references, misspellings, and community-specific language.")
print("These gaps are what the ML-based signals (zero-shot, custom model) should fill.")

EDGE CASES -- where keyword detection struggles:
  [ MISSED] (score: 0.00) Oh great, another trailer that 'doesn't spoil anything' (wink wink)
  [ MISSED] (score: 0.00) If you've read the book, you know exactly what scene they just showed
  [ MISSED] (score: 0.00) bruh they literally showed THAT scene from the comics
  [ MISSED] (score: 0.00) This movie is going to be a killer at the box office
  [ MISSED] (score: 0.00) this traler spoild the hole movie
  [ MISSED] (score: 0.00) They actually showed the snap... why

TAKEAWAY: The keyword detector catches explicit warnings well but misses
sarcasm, indirect references, misspellings, and community-specific language.
These gaps are what the ML-based signals (zero-shot, custom model) should fill.
